In [11]:
source_parquet = '../../food.parquet'

print('TP 1 PARQUET DF:')
print()

import platform
print(f"Python version : {platform.python_version()}")

import pandas as pd
import numpy as np
import datetime as dt

print(f"Pandas version : {pd.__version__}")
print(f"Numpy version : {np.__version__}")

TP 1 PARQUET DF:

Python version : 3.12.10
Pandas version : 3.0.5
Numpy version : 2.5.2


In [12]:
# Outils

from pathlib import Path
import humanize

# Time
import time

class ExecutionTime:
    _start = 0

    def __init__(self):
        self._start = 0

    def start(self):
        self._start = time.time()

    def end(self, text = ""):
        time_exec = time.time() - self._start
        print(f"Execution time {text}: {humanize.precisedelta(time_exec, minimum_unit="microseconds")}")
        print('-----------------------\n')

t = ExecutionTime()

In [13]:
print('PARQUET:')
print()

import pyarrow as pa
import duckdb as db

print(f"Pyarrow version : {pa.__version__}")
print(f"Duckdb version : {db.__version__}")

PARQUET:

Pyarrow version : 25.0.1
Duckdb version : 1.5.5


In [14]:
file_parquet = Path(source_parquet)

print()
print(f"Parquet size: {humanize.naturalsize(file_parquet.stat().st_size)}")
print(f"Last metadata change: {dt.datetime.fromtimestamp((file_parquet.stat().st_ctime)).strftime("%Y-%m-%d %H:%M:%S")}")


Parquet size: 7.7 GB
Last metadata change: 2026-07-29 14:51:32


In [15]:
import pyarrow.parquet as pq

schema = pq.read_schema(source_parquet)

print('Columns name:')

schema_columns_name = []

for field in schema:
    schema_columns_name.append(field.name)
    print(field.name, field.type)

Columns name:
additives_n int32
additives_tags list<element: string>
allergens_tags list<element: string>
brands_tags list<element: string>
brands string
categories string
categories_tags list<element: string>
categories_properties struct<ciqual_food_code: int32, agribalyse_food_code: int32, agribalyse_proxy_food_code: int32>
checkers_tags list<element: string>
ciqual_food_name_tags list<element: string>
cities_tags list<element: string>
code string
compared_to_category string
complete int32
completeness float
correctors_tags list<element: string>
countries_tags list<element: string>
created_t int64
creator string
data_quality_errors_tags list<element: string>
data_quality_info_tags list<element: string>
data_quality_warnings_tags list<element: string>
data_sources_tags list<element: string>
environmental_score_data string
environmental_score_grade string
environmental_score_score int32
environmental_score_tags list<element: string>
editors list<element: string>
emb_codes_tags list<ele

In [16]:
print('LOAD PARQUET FILE')
print()

check_duplicated = ["code"]

use_cols = check_duplicated + [
    "product_name", "brands",
    "lang", "countries_tags",
    "nutriscore_score",
    "nutriscore_grade"
]

t.start()

df_parquet = pd.read_parquet(
    source_parquet,
    engine="pyarrow",
    columns=use_cols
)

display(df_parquet.info())

print(f"Dimensions : {df_parquet.shape}")
print(f"→ {df_parquet.shape[0]} lines, {df_parquet.shape[1]} columns\n")

t.end()

LOAD PARQUET FILE

<class 'pandas.DataFrame'>
RangeIndex: 4636471 entries, 0 to 4636470
Data columns (total 7 columns):
 #   Column            Dtype  
---  ------            -----  
 0   code              str    
 1   product_name      object 
 2   brands            str    
 3   lang              str    
 4   countries_tags    object 
 5   nutriscore_score  float64
 6   nutriscore_grade  str    
dtypes: float64(1), object(2), str(4)
memory usage: 367.6+ MB


None

Dimensions : (4636471, 7)
→ 4636471 lines, 7 columns

Execution time : 15 seconds, 488 milliseconds and 991 microseconds
-----------------------



In [17]:
print('PARQUET SAMPLE DATA')

t.start()

with pd.option_context('display.max_colwidth', None):
    display(df_parquet.head(3))

t.end()

PARQUET SAMPLE DATA


,code,product_name,brands,lang,countries_tags,nutriscore_score,nutriscore_grade
0,0000101209159,"[{'lang': 'main', 'text': 'Véritable pâte à tartiner noisettes chocolat noir'}, {'lang': 'fr', 'text': 'Véritable pâte à tartiner noisettes chocolat noir'}]",Bovetti,fr,[en:france],25.0,e
1,0000105000011,"[{'lang': 'main', 'text': 'Chamomile Herbal Tea'}, {'lang': 'en', 'text': 'Chamomile Herbal Tea'}]",Lagg's,en,[en:united-states],NaN,unknown
2,0000105000042,"[{'lang': 'main', 'text': 'Lagg's, herbal tea, peppermint'}, {'lang': 'en', 'text': 'Lagg's, herbal tea, peppermint'}]",Lagg's,en,[en:united-states],NaN,unknown


Execution time : 9 milliseconds and 796 microseconds
-----------------------



In [19]:
t.start()

print(f"Duplication {check_duplicated}:")
print(df_parquet.duplicated(subset=check_duplicated).value_counts())
print()

print("Valeurs non-renseignées | Taux de remplissage par colonnes:")
missing = pd.DataFrame({
    'total_manquants': df_parquet.isna().sum(),
    '%': 100 - (df_parquet.isna().sum() / len(df_parquet) * 100).round(2)
})
print(missing)
print()

print("Produits vendu en France ([lang].eq(fr):)") #count
print(df_parquet[df_parquet["lang"].eq("fr")].shape[0])
print()

print("Top 10 marques: ")
print(df_parquet.groupby(["brands"]).size().sort_values(ascending=False)[1:11])
print()

print("Quelle part a un Nutri-Score renseigné ?")
print(f"Total: {df_parquet.shape[0]}")
print(f"Manquant: {df_parquet["nutriscore_score"].isna().sum()}")
print(f"Nutri-Score non-renseignés: {(df_parquet["nutriscore_score"].isna().sum() / len(df_parquet) * 100).round(2)}%")

t.end()

Duplication ['code']:
False    4636411
True          60
Name: count, dtype: int64

Valeurs non-renseignées | Taux de remplissage par colonnes:
                  total_manquants       %
code                            0  100.00
product_name                    0  100.00
brands                    1595058   65.60
lang                            4  100.00
countries_tags               9741   99.79
nutriscore_score          3255402   29.79
nutriscore_grade            43447   99.06

Produits vendu en France ([lang].eq(fr):)
1336912

Top 10 marques: 
brands
Carrefour    20732
Coop         14545
Lidl         14243
U            12384
Aldi         12353
BonÀrea      12155
Hacendado    10658
Auchan       10590
Tesco        10503
Delhaize      9918
dtype: int64

Quelle part a un Nutri-Score renseigné ?
Total: 4636471
Manquant: 3255402
Nutri-Score non-renseignés: 70.21%
Execution time : 2 seconds, 869 milliseconds and 411 microseconds
-----------------------

